In [1]:
import pandas as pd
pd.set_option("display.width", 120)

path = "https://raw.githubusercontent.com/Soyoung-Yoon/data_01/main/"


####**로지스틱회귀분석 결과 해석**

**로그-우도**
- 모형의 설명력으로 모델의 성능평가에 사용함
- 로그-우도 : model.llf
- [250514수정] **로그우도가 0에 가까울수록 높은 적합도, 더 작은 음수일수록 낮은 적합도임**

**잔차이탈도(Residual Deviance)**
- 모형이 데이터를 얼마나 잘 설명하는지 평가하는 지표로 0이상의 양수 값을 갖음
- model.deviance, $-2 * (LL_{\text{fitted}} - LL_{\text{saturated}})$
- $LL_{\text{fitted}}$: 현재 적합된(fitted) 모델의 로그 우도
- $LL_{\text{saturated}}$: 포화 모델(saturated model)의 로그 우도
  - 잔차가 없고 최대 우도를 가지는 모델의 로그 우도로 모든 관측값을 완벽하게 설명하는 모델
- 잔차이탈도가 낮을수록 모델이 데이터를 잘 설명
- 잔차이탈도 / 자유도 = 1에 가까우면 적절한 것이고 1보다 크면 과소적합, 1보다 작으면 과대적합 가능성이 있음

**오즈(Odds)**
- 독립변수의 변화에 따라 종속변수가 발생할 확률과 발생하지 않을 확률의 비율(성공과 실패의 확률 비율)
- Odds = P(Y=1) / P(Y=0) = 성공확률 / 실패 확률
- 오즈는 확률의 비율로 항상 0 이상의 값을 갖는다.

**오즈비(Odds Ratio)**
- 특정 독립변수가 1 단위 증가할 때, 종속변수가 발생할 오즈(odds)가 몇 배 증가(또는 감소)하는지를 나타냅니다.
- 특정 변수의 오즈비 : np.exp(model.params['변수명'])
- 특정 변수가 n 증가시 성공의 오즈는 몇 배 증가하는가? : np.exp(model.params['변수명']*5)
- 오즈비가 1보다 크면 Y=1의 가능성이 증가를 의미한다.
- 즉, 독립변수가 증가할수록 종속변수가 발생할 확률이 높아진다.
- 오즈비가 1이면 독립변수는 종속변수에 영향을 미치지 않음
- 오즈비가 1보다 작으면 독립변수가 증가할수록 종속변수 Y=1의 가능성이 감소함(독립변수가 증가할수록 종속변수가 발생할 확률이 낮아진다.)

**유의확률(p-value)**
- 독립변수의 통계적 유의미 판단에 사용함
- p-value가 작을수록 통계적 유의미성은 강한 것임
- 종속변수의 로그 오즈에 통계적으로 유의미한 영향을 미친다.
  - 귀무가설 : 회귀 모델의 기울기가 0이다 즉, 독립변수가 종속변수에 영향을 미치지 않는다.
  - 대립가설 : 회귀 모델의 기울기가 0이 아니다. 즉, 독립변수가 종속변수에 영향을 미친다.
  - 귀무가설 기각 조건: model.pvalues['변수명'] <= 유의수준(0.05)


### Day1. 로지스틱 회귀 (와인 데이터셋)
와인(wine) 데이터를 사용해 로지스틱 회귀를 수행합니다.

In [2]:
df = pd.read_csv(path + "wine01.csv")
print(df.head(3))

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  flavanoids  nonflavanoid_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80        3.06                  0.28   
1    13.20        1.78  2.14               11.2      100.0           2.65        2.76                  0.26   
2    13.16        2.36  2.67               18.6      101.0           2.80        3.24                  0.30   

   proanthocyanins  color_intensity   hue  od280/od315_of_diluted_wines  proline  wine_variety  
0             2.29             5.64  1.04                          3.92   1065.0             0  
1             1.28             4.38  1.05                          3.40   1050.0             0  
2             2.81             5.68  1.03                          3.17   1185.0             0  


In [3]:
# 1-1) 종속변수는 'wine_variety'입니다.
# 범주의 종류 및 개수를 확인
print(df['wine_variety'].value_counts())

wine_variety
1    71
0    59
Name: count, dtype: int64


다음과 같은 로지스틱회귀 모형을 사용한 분류모델을 만들고 결과를 확인합니다.
- wine01.csv 데이터를 사용합니다.
- 모델 생성시 상수항(=절편)을 포함하도록 하며, 규제는 사용하지 않습니다.
- 종속변수 : wine_variety
- 독립변수 : alcohol, color_intensity, proline, flavanoids, malic_acid

In [ ]:
# 1-2) GLM.from_formula() 를 사용해 분석하려고 합니다.
# formula를 작성하고, 로지스틱 회귀모형을 생성합니다.
from statsmodels.api import GLM, add_constant, families
# print(df.shape) # (130, 14)
train = df.iloc[:80, :]
test = df.iloc[80: , :]

formula = "wine_variety ~ alcohol + color_intensity + proline + flavanoids + malic_acid"
model = GLM.from_formula(formula, train).fit()
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           wine_variety   No. Observations:                   80
Model:                            GLM   Df Residuals:                       74
Model Family:                Gaussian   Df Model:                            5
Link Function:               Identity   Scale:                        0.057474
Method:                          IRLS   Log-Likelihood:                 3.8602
Date:                Wed, 05 Nov 2025   Deviance:                       4.2531
Time:                        17:46:15   Pearson chi2:                     4.25
No. Iterations:                     3   Pseudo R-squ. (CS):             0.9134
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           3.4413      0.653     

In [ ]:
#1-3) 모델의 로그-우도를 반올림하여 소수점 아래 3자리까지 출력합니다.
print(round(model.llf, 3)) # 3.860


3.86


In [ ]:
#1-4) 잔차이탈도(Deviance)를 반올림하여 소수점 아래 3자리까지 출력합니다.
print(round(model.deviance, 3)) # 4.253

4.253


In [ ]:
#1-5) 'proline'을 독립변수로 하였을 때의 오즈비(Odds Ratio)는?
# 반올림하여 소수점 아래 3자리까지 출력합니다.
import numpy as np
print(round(np.exp(model.params['proline']), 3)) # 0.999


0.999


In [17]:
#1-6) 'proline'가 3증가하면 오즈는 몇 % 감소 또는 증가하는가?
#(단, 감소율 또는 증가율은 반올림하여 소수점아래 2자리까지 표시합니다.)
# 감소율 = (1 - odds_ratio) * 100
# 증가율 = (odds_ratio - 1) * 100
odds_ratio = np.exp(model.params['proline'] * 3)
# print(odds_ratio)
print(round((odds_ratio - 1)*100, 2)) #  -0.20

-0.2


In [21]:
# 1-7) 아래의 sample을 사용하여 P(Y=1)에 대한 확률을 구하고,
# 반올림하여 소수점 아래 3자리까지 출력하세요.
# sample => alcohol : 13.5, color_intensity: 5.0, proline : 450, flavanoids : 2.8, malic_acid : 1.8
sample = pd.DataFrame({'alcohol': [13.5], 'color_intensity': [5.0], 
                       'proline': [450], 'flavanoids': [2.8],
                       'malic_acid': [1.8]})
# print(sample)
res = model.predict(sample)
print(res.round(3)[0]) # 0.604

0.604


In [23]:
# 1-8) 위 샘플에 대한 odds 를 구하고,
# 반올림하여 소수점 아래 4자리까지 출력하세요.
p_y1 = model.predict(sample)[0]
p_y0 = 1 - p_y1
odds = p_y1 / p_y0
print(odds.round(4)) # 1.5255

1.5255


In [26]:
#1-9) 유의수준 5%하에서, 유의성이 낮은 변수의 개수는 몇 개인가?
res = model.pvalues[1:] > 0.05
print(res)       # color_intensity
print(res.sum()) # 1

alcohol            False
color_intensity     True
proline            False
flavanoids         False
malic_acid         False
dtype: bool
1


In [36]:
#1-10) 아래 샘플에 대한 P(Y=1)에 대한 95% 신뢰구간의 상한은?
# 반올림하여 소수점 아래 4자리까지 출력합니다.
# sample => alcohol : 13.5, color_intensity: 5.0, proline : 850, flavanoids : 2.8, malic_acid : 1.8
sample = pd.DataFrame({'alcohol': [13.5], 'color_intensity': [5.0], 'proline': [850],
                       'flavanoids': [2.8], 'malic_acid': [1.8]})
# print(sample)

result = model.get_prediction(sample)
print(result.summary_frame(alpha=0.05)['mean_ci_upper'].values[0].round(4)) # 0.3989


0.3989


In [37]:
#1-11) 정확도를 구해 반올림하여 소수점 아래 3자리까지 출력합니다.
from sklearn.metrics import accuracy_score
y_true = df['wine_variety']
y_pred = model.predict(sample).round().astype('int32')
acc = accuracy_score(y_true, y_pred)
print(acc.round(3))

ValueError: Found input variables with inconsistent numbers of samples: [130, 1]